# Base Notebook to Run Experiments

## Notebook Setup

### Download Dependencies

In [1]:
!pip install uv
!git clone https://github.com/jorgesilva2407/poc.git /gcr
%cd /gcr
!uv sync --no-dev --no-cache-dir
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 75.1 MB/s eta 0:00:00
Cloning into '/gcr'...
remote: Enumerating objects: 570, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 570 (delta 52), reused 72 (delta 33), pack-reused 472 (from 1)
Receiving objects: 100% (570/570), 73.49 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (280/280), done.
/gcr
Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 186 packages in 0.74ms
Prepared 94 packages in 1m 09s
Installed 94 packages in 226ms
 + absl-py==2.3.1
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.0
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.11.0
 + attrs==25.4.0
 + cachetools==6.2.1
 + certifi==2025.10.5
 + charset-normalizer==3.4.3
 + decorator==5.2.1
 + docstring-parser==0.17.0
 + filelock==3.20.0
 + frozenlist==1.8.0
 + fsspec==2025.9.0
 + gcsfs==2025.9.0
 + google-api-core==2.27.0
 + google-auth==2.41.1
 + google-auth-oauthlib==1

### Import Depedencies
- Import dependencies
- Mount Google Drive
- Define global variables
    - Assert paths exist

In [2]:
import os
import json
import shlex
from enum import Enum
from pathlib import Path
from dataclasses import dataclass

import optuna
from google.colab import drive

drive.mount("/content/drive")

TENSORBOARD_LOG_DIR = "/content/drive/MyDrive/poc/tensorboard"
ARTIFACTS_DIR = "/content/drive/MyDrive/poc/artifacts"
METRIC_FILE = "/tmp/metric.json"
DATA_DIR = "/content/drive/MyDrive/poc/data"
OPTUNA_DIR = "/content/drive/MyDrive/poc/optuna"

Mounted at /content/drive


## Optuna Setup

### Define Hyper Parameter Types

In [3]:
class HParamType(Enum):
    LOGUNIFORM = "loguniform"
    UNIFORM = "uniform"
    INTEGER = "integer"
    CATEGORICAL = "categorical"


@dataclass
class HParam:
    type: HParamType

    def suggest(self, trial, name: str):
        raise NotImplementedError


@dataclass
class Categorical(HParam):
    choices: list[int | float | str]

    def __init__(self, choices: list[int | float | str]):
        super().__init__(HParamType.CATEGORICAL)
        self.choices = choices

    def suggest(self, trial, name: str):
        return trial.suggest_categorical(name, self.choices)


@dataclass
class Uniform(HParam):
    low: float
    high: float

    def __init__(self, low: float, high: float):
        super().__init__(HParamType.UNIFORM)
        self.low = low
        self.high = high

    def suggest(self, trial, name: str):
        return trial.suggest_uniform(name, self.low, self.high)


@dataclass
class LogUniform(HParam):
    low: float
    high: float

    def __init__(self, low: float, high: float):
        super().__init__(HParamType.LOGUNIFORM)
        self.low = low
        self.high = high

    def suggest(self, trial, name: str):
        return trial.suggest_loguniform(name, self.low, self.high)


@dataclass
class Integer(HParam):
    low: int
    high: int

    def __init__(self, low: int, high: int):
        super().__init__(HParamType.INTEGER)
        self.low = low
        self.high = high

    def suggest(self, trial, name: str):
        return trial.suggest_int(name, self.low, self.high)

### Helper Function to get a set of Hyper Parameters to use

In [4]:
def suggest_params(trial, search_space):
    params = {}
    for name, hparam in search_space.items():
        params[name] = hparam.suggest(trial, name)
    return params

### Model Runner

In [5]:
def run_model(
    model_name: str,
    params: dict,
    dataset: str,
    all_csv: str,
    train_csv: str,
    val_csv: str,
    test_csv: str,
) -> float:
    tensorboard_log_dir = Path(TENSORBOARD_LOG_DIR) / dataset
    artifacts_dir = Path(ARTIFACTS_DIR) / dataset

    os.makedirs(tensorboard_log_dir, exist_ok=True)
    os.makedirs(artifacts_dir, exist_ok=True)

    cmd = [
        "uv",
        "run",
        "python",
        "-u",
        "main.py",
        "--model",
        model_name,
        "--logger",
        "TensorBoard",
        "--tensorboard-log-dir",
        str(tensorboard_log_dir),
        "--artifacts-saver",
        "Local",
        "--optuna-metric-file",
        METRIC_FILE,
        "--local-artifacts-path",
        str(artifacts_dir),
        "--experiment-tracker",
        "Optuna",
        "--batch-size",
        64,
        "--all-interactions-csv",
        all_csv,
        "--train-interactions-csv",
        train_csv,
        "--validation-interactions-csv",
        val_csv,
        "--test-interactions-csv",
        test_csv,
    ]

    for param, value in params.items():
        cmd.extend([param, str(value)])

    cmd = [str(c) for c in cmd]

    shell_cmd = shlex.join(cmd)
    print(shell_cmd)

    !{shell_cmd}

    metric = None
    with open(METRIC_FILE, "r") as f:
        metric = json.load(f)
    return metric["NDCG@5"]

### Define the Objective of the Optuna Study

In [6]:
def objective_factory(model_name: str, search_space: dict, dataset: str):
    def objective(trial):
        params = suggest_params(trial, search_space)

        data_dir = Path(DATA_DIR) / dataset
        all_csv = data_dir / "all_interactions.csv"
        train_csv = data_dir / "split" / "train.csv"
        val_csv = data_dir / "split" / "val.csv"
        test_csv = data_dir / "split" / "test_neg_samples.csv"

        paths = [data_dir, all_csv, train_csv, val_csv, test_csv]
        missing_paths = [path for path in paths if not path.exists()]

        if missing_paths:
            raise ValueError(f"Missing paths: {missing_paths}")

        result = run_model(
            model_name, params, dataset, all_csv, train_csv, val_csv, test_csv
        )
        return result

    return objective

### Run an Optuna Study

In [7]:
def optimize_model(
    model_name: str, search_space: dict, dataset: str, n_trials: int = 20
):
    db_path = Path(OPTUNA_DIR) / dataset / f"{model_name}.db"
    os.makedirs(db_path.parent, exist_ok=True)

    study = optuna.create_study(
        study_name=f"{model_name}Optimization",
        direction="maximize",
        storage=f"sqlite:///{db_path}",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    completed = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    remaining = n_trials - completed

    if remaining <= 0:
        print(f"Study already has {completed}/{n_trials} trials.")
        print(f"Best value: {study.best_value}")
        print(f"Best params: {study.best_params}")
        return study

    print(
        f"🔄 Resuming study: {completed} trials done, {remaining} remaining to reach {n_trials}"
    )

    objective = objective_factory(model_name, search_space, dataset)
    study.optimize(objective, n_trials=remaining)

    print("\n✅ Optimization complete!")
    print(f"Best value: {study.best_value}")
    print(f"Best params: {study.best_params}")

    return study

## Run the Study

Update the model, dataset and search space as needed

In [ ]:
model = "GCR"
dataset = "clothing"

user_item_embedding_dims = [16, 32, 64, 128, 256]
event_embedding_dims = [16, 32, 64, 128, 256]
hidden_dims = [16, 32, 64, 128, 256]
num_neighbors = [3, 5, 7]
reg_weights = [1e-4, 1e-3, 1e-2, 1e-1]
learning_rates = [1e-4, 5e-4, 1e-3, 1e-2]
weight_decays = [1e-6, 1e-5, 1e-4, 1e-3]
dropout = [0.0, 0.2, 0.5]

search_space = {
    "--user-item-embedding-dim": Categorical(choices=user_item_embedding_dims),
    "--event-embedding-dim": Categorical(choices=event_embedding_dims),
    "--hidden-dim": Categorical(choices=hidden_dims),
    "--num-neighbors": Categorical(choices=num_neighbors),
    "--reg-weight": Categorical(choices=reg_weights),
    "--learning-rate": Categorical(choices=learning_rates),
    "--weight-decay": Categorical(choices=weight_decays),
    "--dropout-rate": Categorical(choices=dropout),
}

In [ ]:
optimize_model(model, search_space, dataset, n_trials=20)

[I 2025-11-21 15:01:28,966] A new study created in RDB with name: GCROptimization


🔄 Resuming study: 0 trials done, 20 remaining to reach 20
uv run python -u main.py --model GCR --logger TensorBoard --tensorboard-log-dir /content/drive/MyDrive/poc/tensorboard/clothing --artifacts-saver Local --optuna-metric-file /tmp/metric.json --local-artifacts-path /content/drive/MyDrive/poc/artifacts/clothing --experiment-tracker Optuna --batch-size 64 --all-interactions-csv /content/drive/MyDrive/poc/data/clothing/all_interactions.csv --train-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/train.csv --validation-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/val.csv --test-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/test_neg_samples.csv --user-item-embedding-dim 32 --event-embedding-dim 64 --hidden-dim 32 --num-neighbors 7 --reg-weight 0.01 --learning-rate 0.01 --weight-decay 0.0001
Installed 88 packages in 110ms
Ignored arguments: []
Using device: cuda
Epoch 1 - Training: 100% 616/616 [02:17<00:00,  4.49it/s]
Epoch 1 - Valida

[I 2025-11-21 15:48:07,849] Trial 0 finished with value: 0.05612213909626007 and parameters: {'--user-item-embedding-dim': 32, '--event-embedding-dim': 64, '--hidden-dim': 32, '--num-neighbors': 7, '--reg-weight': 0.01, '--learning-rate': 0.01, '--weight-decay': 0.0001}. Best is trial 0 with value: 0.05612213909626007.


uv run python -u main.py --model GCR --logger TensorBoard --tensorboard-log-dir /content/drive/MyDrive/poc/tensorboard/clothing --artifacts-saver Local --optuna-metric-file /tmp/metric.json --local-artifacts-path /content/drive/MyDrive/poc/artifacts/clothing --experiment-tracker Optuna --batch-size 64 --all-interactions-csv /content/drive/MyDrive/poc/data/clothing/all_interactions.csv --train-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/train.csv --validation-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/val.csv --test-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/test_neg_samples.csv --user-item-embedding-dim 256 --event-embedding-dim 16 --hidden-dim 128 --num-neighbors 3 --reg-weight 0.01 --learning-rate 0.0001 --weight-decay 0.001
Ignored arguments: []
Using device: cuda
Epoch 1 - Training: 100% 616/616 [02:12<00:00,  4.64it/s]
Epoch 1 - Validation: 100% 62/62 [00:23<00:00,  2.68it/s]
Epoch 1: Train Loss = 0.7941, Val Loss = 0.6

[I 2025-11-21 16:34:44,379] Trial 1 finished with value: 0.18568512797355652 and parameters: {'--user-item-embedding-dim': 256, '--event-embedding-dim': 16, '--hidden-dim': 128, '--num-neighbors': 3, '--reg-weight': 0.01, '--learning-rate': 0.0001, '--weight-decay': 0.001}. Best is trial 1 with value: 0.18568512797355652.


...

uv run python -u main.py --model GCR --logger TensorBoard --tensorboard-log-dir /content/drive/MyDrive/poc/tensorboard/clothing --artifacts-saver Local --optuna-metric-file /tmp/metric.json --local-artifacts-path /content/drive/MyDrive/poc/artifacts/clothing --experiment-tracker Optuna --batch-size 64 --all-interactions-csv /content/drive/MyDrive/poc/data/clothing/all_interactions.csv --train-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/train.csv --validation-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/val.csv --test-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/test_neg_samples.csv --user-item-embedding-dim 32 --event-embedding-dim 64 --hidden-dim 256 --num-neighbors 7 --reg-weight 0.001 --learning-rate 0.0001 --weight-decay 1e-06
Ignored arguments: []
Using device: cuda
Epoch 1 - Training: 100% 616/616 [02:17<00:00,  4.49it/s]
Epoch 1 - Validation: 100% 62/62 [00:20<00:00,  3.09it/s]
Epoch 1: Train Loss = 0.7046, Val Loss = 0.6

[I 2025-11-21 18:54:36,167] Trial 4 finished with value: 0.15910077095031738 and parameters: {'--user-item-embedding-dim': 32, '--event-embedding-dim': 64, '--hidden-dim': 256, '--num-neighbors': 7, '--reg-weight': 0.001, '--learning-rate': 0.0001, '--weight-decay': 1e-06}. Best is trial 1 with value: 0.18568512797355652.


uv run python -u main.py --model GCR --logger TensorBoard --tensorboard-log-dir /content/drive/MyDrive/poc/tensorboard/clothing --artifacts-saver Local --optuna-metric-file /tmp/metric.json --local-artifacts-path /content/drive/MyDrive/poc/artifacts/clothing --experiment-tracker Optuna --batch-size 64 --all-interactions-csv /content/drive/MyDrive/poc/data/clothing/all_interactions.csv --train-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/train.csv --validation-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/val.csv --test-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/test_neg_samples.csv --user-item-embedding-dim 256 --event-embedding-dim 64 --hidden-dim 64 --num-neighbors 3 --reg-weight 0.01 --learning-rate 0.001 --weight-decay 0.0001
Ignored arguments: []
Using device: cuda
Epoch 1 - Training: 100% 616/616 [02:12<00:00,  4.65it/s]
Epoch 1 - Validation: 100% 62/62 [00:22<00:00,  2.78it/s]
Epoch 1: Train Loss = 0.6279, Val Loss = 0.73

[I 2025-11-21 19:58:51,454] Trial 5 finished with value: 0.09482875466346741 and parameters: {'--user-item-embedding-dim': 256, '--event-embedding-dim': 64, '--hidden-dim': 64, '--num-neighbors': 3, '--reg-weight': 0.01, '--learning-rate': 0.001, '--weight-decay': 0.0001}. Best is trial 1 with value: 0.18568512797355652.


uv run python -u main.py --model GCR --logger TensorBoard --tensorboard-log-dir /content/drive/MyDrive/poc/tensorboard/clothing --artifacts-saver Local --optuna-metric-file /tmp/metric.json --local-artifacts-path /content/drive/MyDrive/poc/artifacts/clothing --experiment-tracker Optuna --batch-size 64 --all-interactions-csv /content/drive/MyDrive/poc/data/clothing/all_interactions.csv --train-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/train.csv --validation-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/val.csv --test-interactions-csv /content/drive/MyDrive/poc/data/clothing/split/test_neg_samples.csv --user-item-embedding-dim 64 --event-embedding-dim 32 --hidden-dim 64 --num-neighbors 7 --reg-weight 0.0001 --learning-rate 0.0005 --weight-decay 1e-05
Ignored arguments: []
Using device: cuda
Epoch 1 - Training: 100% 616/616 [02:16<00:00,  4.51it/s]
Epoch 1 - Validation: 100% 62/62 [00:19<00:00,  3.10it/s]
Epoch 1: Train Loss = 0.6945, Val Loss = 0.6